In [ ]:
"""
Versão 1.2:
Aqui testamos a eficiência do sistema utilizando a divisão de chunks por contexto ao invés de tamanho 
fixo de caracteres, utilizando o pdfplumber para identificar separadores de capítulos e seções
"""

# Preparação dos Documentos

In [1]:
import pdfplumber
#import pytesseract
#import pdf2image
import pandas as pd
# import tiktoken
import chromadb
import openai
import json
import os
import re
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from typing import List, Dict
from statistics import mean
from dotenv import load_dotenv, find_dotenv

In [22]:
CHUNK_SIZE = 1000
OFFSET = 200

# tokenizer = tiktoken.get_encoding("cl100k_base")
load_dotenv(find_dotenv())
openai_client = OpenAI(api_key= os.environ["OPENAI_API_KEY"])

# Conexão com o cliente do banco de dados ChromaDB
chromadb_path = "G:/Drives compartilhados/RISCO E COMPLIANCE/Relatórios de Risco/Risco/FIDCS/SCRIPTS_RISCO/Projeto IA/Base ChromaDB/"
chroma_client = chromadb.PersistentClient(path= chromadb_path)
collection    = chroma_client.get_or_create_collection(name= "Regulamentos_V1.2")

In [3]:
# Checa todo o conteúdo inserido no ChromaDB
collection.get()

{'ids': ['Regulamento Poupacred II.pdf_chunk_0',
  'Regulamento Poupacred II.pdf_chunk_1',
  'Regulamento Poupacred II.pdf_chunk_2',
  'Regulamento Poupacred II.pdf_chunk_3',
  'Regulamento Poupacred II.pdf_chunk_4',
  'Regulamento Poupacred II.pdf_chunk_5',
  'Regulamento Poupacred II.pdf_chunk_6',
  'Regulamento Poupacred II.pdf_chunk_7',
  'Regulamento Poupacred II.pdf_chunk_8',
  'Regulamento Poupacred II.pdf_chunk_9',
  'Regulamento Poupacred II.pdf_chunk_10',
  'Regulamento Poupacred II.pdf_chunk_11',
  'Regulamento Poupacred II.pdf_chunk_12',
  'Regulamento Poupacred II.pdf_chunk_13',
  'Regulamento Poupacred II.pdf_chunk_14',
  'Regulamento Poupacred II.pdf_chunk_15',
  'Regulamento Poupacred II.pdf_chunk_16',
  'Regulamento Poupacred II.pdf_chunk_17',
  'Regulamento Poupacred II.pdf_chunk_18',
  'Regulamento Poupacred II.pdf_chunk_19',
  'Regulamento Poupacred II.pdf_chunk_20',
  'Regulamento Poupacred II.pdf_chunk_21',
  'Regulamento Poupacred II.pdf_chunk_22',
  'Regulamento

In [23]:
# def count_tokens(text):
#     return len(tokenizer.encode(text))

def split_document(document_text):
    documents = []
    for i in range(0, len(document_text), CHUNK_SIZE):
        start = i
        end = i + CHUNK_SIZE
        if start != 0:
            start = start - OFFSET
            end =  end - OFFSET
        documents.append(document_text[start: end])
    return documents

# Obtenção dos embeddings usando Langchain
# def get_embedding_langchain(text):

#     embedding = OpenAIEmbeddings(
#         model= "text-embedding-3-small",
#         chunk_size= CHUNK_SIZE
#     )

#     emb = embedding.embed_query(text)

#     return emb

# Obtenção dos embeddings usando diretamente a API da OpenAI
def get_embedding_openai(text, client):

    emb = client.embeddings.create(
        input= text,
        model= "text-embedding-3-small"
    )

    return emb.data[0].embedding

# Extração de texto dos arquivos PDF
# Texto é dividido em chunks baseados em títulos detectados por tamanho de fonte, negrito e palavras-chave.
def extract_structured_chunks(pdf_path: str, max_chunk_words: int = 1500) -> List[Dict]:
    
    chunks = []
    current_chunk = {"chapter": None, "section": None, "content": ""}

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            chars = page.chars  # lista de caracteres com fonte, tamanho, posição
            if not chars:
                continue

            # ----- Calcular estatísticas da página 
            font_sizes = [float(c["size"]) for c in chars]
            avg_font_size = mean(font_sizes)
            big_font_threshold = avg_font_size * 1.2  # 20% maior que a média

            # ----- Extrair texto linha a linha 
            lines = page.extract_text().split("\n") if page.extract_text() else []
            for line in lines:
                # Coletar caracteres da linha
                line_chars = [c for c in chars if _is_char_in_line(c, line)]
                if not line_chars:
                    continue

                # Detectar tamanho médio e fontes da linha
                line_font_size = mean([float(c["size"]) for c in line_chars])
                fontnames = set(c["fontname"] for c in line_chars)
                is_bold = any("Bold" in f or "Black" in f or "Heavy" in f for f in fontnames)
                is_big = line_font_size >= big_font_threshold

                # Detectar se "capitulo" está centralizado
                x0 = min(c["x0"] for c in line_chars)
                x1 = max(c["x1"] for c in line_chars)
                word_center = (x0 + x1) / 2
                page_center = page.width / 2
                is_centered = abs(word_center - page_center) < (page.width * 0.15)

                # ----- Detectar se é um título
                if (is_big or is_bold or is_centered) and re.match(r"(?i)\s*(cap[ií]tulo)\b", line):

                    # Novo capítulo ou seção -> salvar chunk anterior
                    if current_chunk["content"].strip():
                        chunks.append(current_chunk.copy())

                    # Definir novo título
                    if re.search(r"(?i)cap[ií]tulo", line):
                        current_chunk = {"chapter": line.strip(), "section": None, "content": ""}
                        match_section = re.search(r"^\s*((\d+\.){1,}\d*)", line, re.IGNORECASE)
                    elif match_section:
                        current_chunk["section"] = match_section.group(1)
                    else:
                        current_chunk["content"] = line.strip()
                    continue

                # ----- Caso contrário, adicionar o texto normal 
                current_chunk["content"] += " " + line.strip()

                # ----- Se o chunk ficar muito grande, quebrar 
                if len(current_chunk["content"].split()) > max_chunk_words:
                    chunks.append({
                        "chapter": current_chunk["chapter"],
                        "section": current_chunk["section"],
                        "content": current_chunk["content"].strip()
                    })
                    current_chunk["content"] = ""

        # ----- Adicionar último chunk 
        if current_chunk["content"].strip():
            chunks.append(current_chunk.copy())

    return chunks

# Função auxiliar que verifica se um caractere pertence a uma linha extraída
# Heurística simples: confere se o caractere aparece na linha.
# (não é 100% exato, mas suficiente para manter a correlação)
def _is_char_in_line(char, line_text):
    return any(c.lower() in line_text.lower() for c in char["text"] if c.strip())

# Extração das tabelas dos arquivos PDF
# Tabelas serão extraídas diferentemente dos blocos de texto
def extract_tables_from_pdf(pdf_path):
    data_list = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table in tables:
                df = pd.DataFrame(table[1:], columns=table[0])  # Usa a primeira linha como cabeçalho
                data_list.append(df)

    return data_list

# Guarda os dados extraidos dos documentos no ChromaDB
def insert_text_chromadb(data, source, client):

    for idx, chunk in enumerate(data):

        if chunk['section']:
            data     = f"{chunk['section']} \n {chunk['content']}" 
            metadata = f"{source}_{chunk['chapter']}_{chunk['section']}" 
            docs     = f"{chunk['section']} \n {chunk['content']}" 
        else:
            metadata = f"{source}_{chunk['chapter']}"
            docs     = chunk['content']
            data     = chunk['content']

        embedding = get_embedding_openai(data, client)

        collection.add(
            ids=[f"{source}_chunk_{idx}"],
            documents=[docs],
            metadatas=[{"source": metadata, "chunk_index": idx}],
            embeddings=[embedding]
        )

def insert_tables_chromadb(data, source, client):

    chunks = split_document(data)

    print(chunks)

    for idx, chunk in enumerate(chunks):

        embedding = get_embedding_openai(chunk, client)

        collection.add(
            ids=[f"{source}_chunk_{idx}"],
            documents=[chunk],
            metadatas=[{"source": source, "chunk_index": idx}],
            embeddings=[embedding]
        )

In [24]:
# def run():
print("Preparando Documentos...")
data_path = 'G:/Drives compartilhados/GESTAO/_Operacional/Planilhas Gestão/Scripts/temp/temp_regulamentos/Teste 1.1'

documents_names = [f for f in os.listdir(data_path) if f.endswith('.pdf')]
documents_names_size = len(documents_names)

for i, document_name in enumerate(documents_names):

    print(f"{i+1}/{documents_names_size}: {document_name}")

    doc_path = os.path.join(data_path, document_name)

    document_data = extract_structured_chunks(os.path.join(data_path, document_name))
    insert_text_chromadb(document_data, document_name, openai_client)

    tables = extract_tables_from_pdf(doc_path)
    if tables:
        for idx, df in enumerate(tables):
            json_data = df.to_json(orient= "records")
            insert_tables_chromadb(json_data, document_name, openai_client)

# if __name__ == "__main__":
#     run()
#     pass

Preparando Documentos...
1/1: Regulamento Poupacred II.pdf
['[{"Patrim\\u00f4nio L\\u00edquido":"De R$ 0 a R$ 250.000.000,00","Taxa (a.a.)":"0,13%"},{"Patrim\\u00f4nio L\\u00edquido":"De R$ 250.000.000,01 a R$ 500.000.000,00","Taxa (a.a.)":"0,11%"},{"Patrim\\u00f4nio L\\u00edquido":"De R$ 500.000.000,01 a R$ 750.000.000,00","Taxa (a.a.)":"0,09%"},{"Patrim\\u00f4nio L\\u00edquido":"R$ 750.000.000,01 a R$ 1.000.000.000,00","Taxa (a.a.)":"0,07%"},{"Patrim\\u00f4nio L\\u00edquido":"Acima de R$ 1.000.000.000,01","Taxa (a.a.)":"0,05%"}]']


In [25]:
document_data

[{'chapter': None,
  'section': None,
  'content': ' REGULAMENTO DO FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS POUPACRED II RESPONSABILIDADE LIMITADA CNPJ/MF Nº 57.377.181/0001-97 Vigente a partir de 24 janeiro de 2025 SUMÁRIO PARTE GERAL ................................................................................................................................ 4'},
 {'chapter': 'CAPÍTULO V – DOS DEMAIS PRESTADORES DE SERVIÇOS .................................................... 13',
  'section': None,
  'content': ' DO FUNDO .............................................................................................................................. 13'},
 {'chapter': 'CAPÍTULO XIV – DO FORO ...................................................................................................... 23',
  'section': None,
  'content': ' ANEXO I ....................................................................................................................................... 24 CA

In [33]:
# Checa todo o conteúdo inserido no ChromaDB
collection.get()

{'ids': ['Regulamento Poupacred II.pdf_chunk_0',
  'Regulamento Poupacred II.pdf_chunk_1',
  'Regulamento Poupacred II.pdf_chunk_2',
  'Regulamento Poupacred II.pdf_chunk_3',
  'Regulamento Poupacred II.pdf_chunk_4',
  'Regulamento Poupacred II.pdf_chunk_5',
  'Regulamento Poupacred II.pdf_chunk_6',
  'Regulamento Poupacred II.pdf_chunk_7',
  'Regulamento Poupacred II.pdf_chunk_8',
  'Regulamento Poupacred II.pdf_chunk_9',
  'Regulamento Poupacred II.pdf_chunk_10',
  'Regulamento Poupacred II.pdf_chunk_11',
  'Regulamento Poupacred II.pdf_chunk_12',
  'Regulamento Poupacred II.pdf_chunk_13',
  'Regulamento Poupacred II.pdf_chunk_14',
  'Regulamento Poupacred II.pdf_chunk_15',
  'Regulamento Poupacred II.pdf_chunk_16',
  'Regulamento Poupacred II.pdf_chunk_17',
  'Regulamento Poupacred II.pdf_chunk_18',
  'Regulamento Poupacred II.pdf_chunk_19',
  'Regulamento Poupacred II.pdf_chunk_20',
  'Regulamento Poupacred II.pdf_chunk_21',
  'Regulamento Poupacred II.pdf_chunk_22',
  'Regulamento

In [ ]:
# def delete_existing_chunks(source):
#     existing_chunks = collection.get()["ids"]

#     for chunk_id in existing_chunks:
#         if chunk_id.startswith(source):
#             collection.delete(ids=[chunk_id])
#             print(f"🗑️ Chunk {chunk_id} deletado.")

# Exemplo: Deletar apenas os chunks do documento "CONCRETO-CONSIGNADO"
# delete_existing_chunks("Regulamento Poupacred II")

🗑️ Chunk Regulamento Poupacred II.pdf_chunk_0 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_1 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_2 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_3 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_4 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_5 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_6 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_7 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_8 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_9 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_10 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_11 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_12 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_13 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_14 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_15 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chunk_16 deletado.
🗑️ Chunk Regulamento Poupacred II.pdf_chu

# Implementação da RAG

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

def detect_document_mention(query, available_sources):
    for source in available_sources:
        if re.search(re.escape(source), query, re.IGNORECASE):  # Busca pelo nome do documento na pergunta
            return source
    return None  # Retorna None se nenhum documento for encontrado

# Função que pesquisa pelos documentos relevantes baseando-se no contexto da pergunta
def search_document(question, client):

    print("Pesquisando documentos relevantes...")

    query_embedding = get_embedding_openai(question, client)

    results = collection.query(
        query_embeddings= [query_embedding],
        n_results= 20
    )

    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        print(f"\nFonte: {meta['source']} - Tipo: {meta['type']}")
        print(doc[:1000])  

def ask_llm(query, client):

    stored_data = collection.get(include=["metadatas"])
    available_sources = {meta["source"] for meta in stored_data["metadatas"]}  # Obtém nomes únicos dos documentos

    # Verifica se o usuário mencionou um documento específico
    specific_document = detect_document_mention(query, available_sources)

    # Definição do filtro para busca no ChromaDB
    search_filter = {"source": specific_document} if specific_document else None  # Filtra apenas se um documento for citado

    # Busca informações no ChromaDB
    embedding = get_embedding_openai(query, client)

    results = collection.query(
        query_embeddings=[embedding],
        n_results=20,
        where= search_filter
    )

    # Organiza os resultados agrupando por 'source' (nome do documento)
    retrieved_docs = results["documents"][0]
    metadatas      = results["metadatas"][0]

    grouped_docs = {}  # Dicionário para agrupar os chunks pelo nome do documento

    for doc, meta in zip(retrieved_docs, metadatas):
        source = meta["source"]  # Nome do documento original
        if source not in grouped_docs:
            grouped_docs[source] = []
        grouped_docs[source].append(doc)  # Adiciona o chunk ao documento correspondente

    # Criar um contexto consolidado por documento
    context = "\n\n".join([
        f"🔹 Documento: {source}\n" + "\n".join(chunks)
        for source, chunks in grouped_docs.items()
    ])

    modelo = ChatOpenAI(model = "gpt-4o-mini", temperature= 0, max_tokens= 4096)

    prompt = f"""
    Você é um assistente especializado em responder perguntas sobre vários documentos PDF armazenados em um banco de dados.
    Cada documento está dividido em múltiplos IDs, diferenciando-se apenas pelo número ao final.
    Documentos cujos IDs tenham nomes diferente não possuem relação alguma um com o outro.
    Quando for dado o nome de um fundo de investimento, forneça informação contida apenas naquele chunk. Não misture informação de diferentes documentos
    Considere os seguintes documentos: 
    {context}
    
    Caso não seja mencionado nenhum nome de fundo, responda considerando **todos** os documentos mencionados acima.
    """
    #Se não for encontrado nenhum documento semelhante ao nome dado, informe.
    messages=[
    SystemMessage(content= prompt),
    HumanMessage( content= query)
    ]

    answer = modelo.invoke(messages)

    return answer.content
    # return response["choices"][0]["message"]["content"]

# Aplicação do LLM

In [32]:
# question = """
# Quais são os gestores dos fundos Concreto e Concrédito II?
# """

question = """
Como estão estruturadas as classes de cotas do FIDC Poupacred II?
"""

print(ask_llm(question, openai_client))

O FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS POUPACRED II possui uma única classe de Cotas, que é dividida em três subclasses:

1. **Cotas Seniores**: Têm prioridade de amortização e/ou resgate em relação às Cotas Subordinadas Mezanino e às Cotas Subordinadas Júnior.

2. **Cotas Subordinadas Mezanino**: Subordinam-se às Cotas Seniores e têm prioridade em relação às Cotas Subordinadas Júnior.

3. **Cotas Subordinadas Júnior**: Subordinam-se às Cotas Seniores e às Cotas Subordinadas Mezanino para efeito de amortização, resgate e distribuição dos rendimentos da Classe.

Essas subclasses têm características e condições específicas para amortização e resgate, conforme detalhado nos regulamentos do fundo.


## Aplicação do LLM (ANTIGO)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

modelo_antigo = ChatOpenAI(model = "gpt-4o-mini", temperature= 0, max_tokens= 4096)

prompt = """Você é um assistente de IA que responde as dúvidas dos usuários com bases nos documentos abaixo.
Os documentos abaixo apresentam as fontes atualizadas e devem ser consideradas como verdade. Sempre que uma
pergunta oferecer o nome de um FIDC, você deve procurar pela melhor correspondência deste nome na lista de documentos.
Cite a fonte quando fornecer a informação.
Documentos:
{documents}
""" #Caso o nome citado na pergunta não exista na lista de documentos, responda que ele não foi encontrado.

prompt = prompt.format(documents= documents_str)

messages=[
  SystemMessage(content= prompt),
  HumanMessage(content= question)
]

In [ ]:
answer = modelo_antigo.invoke(messages)

In [ ]:
answer.content

"Com base nos documentos disponíveis, não encontrei menções aos termos 'Saque aniversário', 'FGTS', 'INSS', 'Consignado' ou 'Saque-Aniversário FGTS'. Portanto, não há documentos que incluam qualquer um desses termos."

In [38]:
from chromadb import Client
from chromadb.config import Settings

collection    = chroma_client.get_or_create_collection(name= "Regulamentos_V1.1")

# Obtenha todos os documentos
all_docs = collection.get()

# Filtre pelo prefixo do ID
prefix = "Regulamento Poupacred II"
filtered_docs = [
    {
        "id": _id,
        "document": doc,
        "metadata": meta
    }
    for _id, doc, meta in zip(all_docs["ids"], all_docs["documents"], all_docs["metadatas"])
    if _id.startswith(prefix)
]

print(f"Encontrados {len(filtered_docs)} chunks:")
for d in filtered_docs:
    print("-", d["id"])


Encontrados 210 chunks:
- Regulamento Poupacred II.pdf_chunk_0
- Regulamento Poupacred II.pdf_chunk_1
- Regulamento Poupacred II.pdf_chunk_2
- Regulamento Poupacred II.pdf_chunk_3
- Regulamento Poupacred II.pdf_chunk_4
- Regulamento Poupacred II.pdf_chunk_5
- Regulamento Poupacred II.pdf_chunk_6
- Regulamento Poupacred II.pdf_chunk_7
- Regulamento Poupacred II.pdf_chunk_8
- Regulamento Poupacred II.pdf_chunk_9
- Regulamento Poupacred II.pdf_chunk_10
- Regulamento Poupacred II.pdf_chunk_11
- Regulamento Poupacred II.pdf_chunk_12
- Regulamento Poupacred II.pdf_chunk_13
- Regulamento Poupacred II.pdf_chunk_14
- Regulamento Poupacred II.pdf_chunk_15
- Regulamento Poupacred II.pdf_chunk_16
- Regulamento Poupacred II.pdf_chunk_17
- Regulamento Poupacred II.pdf_chunk_18
- Regulamento Poupacred II.pdf_chunk_19
- Regulamento Poupacred II.pdf_chunk_20
- Regulamento Poupacred II.pdf_chunk_21
- Regulamento Poupacred II.pdf_chunk_22
- Regulamento Poupacred II.pdf_chunk_23
- Regulamento Poupacred II